In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import tqdm
from src.language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from src.language_models.model import RNNModel as lstm
from src.language_models.utils import move_to_device
import random
import pandas as pd
from collections import defaultdict
import numpy as np


In [19]:
batch_size = 1024
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)
checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt"  # Replace with your checkpoint path


In [20]:
model = lstm('LSTM', 50001, 200, 650, 2, 0.2, False)
with open(checkpoint_path, 'rb') as f:
    state_dict = torch.load(f, map_location='cuda' if device =='cuda' else 'cpu')
    model.load_state_dict(state_dict)
model.to(device)
model.eval() 

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 200)
  (rnn): LSTM(200, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

In [21]:
nounpp = '//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt'

In [22]:
class NounPPDataset(Dataset):
    def __init__(self, nounpp_file, dictionary):
        self.sentences = []
        self.conditions = []
        self.correct = []
        self.wrong = []
        self.encoded_sentences=[]
        self.encoded_correct = []
        self.encoded_wrong = []
        self.dictionary = dictionary

        with open(nounpp_file, "r") as f:
            for line in f:
                line = line.split()
                sentence = line[1:7]
                condition = " ".join(line[7:9])
                wrong = line[9]
                correct = line[6]
                encoded_sentence = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence]
                encoded_correct = self.dictionary.word2idx.get(correct, self.dictionary.word2idx.get("<unk>"))
                encoded_wrong = self.dictionary.word2idx.get(wrong, self.dictionary.word2idx.get("<unk>"))
                
                self.sentences.append(sentence)
                self.conditions.append(condition)
                self.correct.append(correct)
                self.wrong.append(wrong)
                self.encoded_sentences.append(encoded_sentence)
                self.encoded_correct.append(encoded_correct)
                self.encoded_wrong.append(encoded_wrong)

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return {
            "sentence": self.sentences[idx],
            "encoded_sentence": torch.tensor(self.encoded_sentences[idx], dtype=torch.long),
            "correct": self.correct[idx],
            "encoded_correct":torch.tensor(self.encoded_correct[idx], dtype = torch.long),
            "wrong":self.wrong[idx],
            "encoded_wrong":torch.tensor(self.encoded_wrong[idx], dtype = torch.long),
            "condition": self.conditions[idx],
        }

In [23]:
test_dataset = NounPPDataset(nounpp, dictionary)

In [25]:
def collate_fn(batch):
    """Custom collate function to properly handle sentences as lists of strings."""
    sentences = [item['sentence'] for item in batch]  # Keep lists of words as they are
    encoded_sentences = torch.stack([item['encoded_sentence'] for item in batch])  # Stack tensors
    encoded_correct = torch.stack([item['encoded_correct'] for item in batch])
    encoded_wrong = torch.stack([item['encoded_wrong'] for item in batch])
    correct = [item['correct'] for item in batch]
    wrong = [item['wrong'] for item in batch]
    conditions = [item['condition'] for item in batch]
    
    return {
        "sentence": sentences,  
        "encoded_sentence": encoded_sentences,
        "correct": correct,
        "encoded_correct": encoded_correct,
        "wrong": wrong,
        "encoded_wrong": encoded_wrong,
        "condition": conditions
    }

In [26]:
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn)

In [28]:
init_sentence = " ".join(["In service , the aircraft was operated by a crew of five and could accommodate either 30 paratroopers , 32 <unk> and 28 sitting casualties , or 50 fully equipped troops . <eos>",
                    "He even speculated that technical classes might some day be held \" for the better training of workmen in their several crafts and industries . <eos>",
                    "After the War of the Holy League in 1537 against the Ottoman Empire , a truce between Venice and the Ottomans was created in 1539 . <eos>",
                    "Moore says : \" Tony and I had a good <unk> and off-screen relationship , we are two very different people , but we did share a sense of humour \" . <eos>",
                    "<unk> is also the basis for online games sold through licensed lotteries . <eos>"])

In [29]:
def feed_input(model, hidden, w):
    inp = torch.autograd.Variable(torch.LongTensor([[dictionary.word2idx[w]]]))
    out, hidden = model(inp, hidden)
    return out, hidden
def feed_sentence(model, h, sentence):
    outs = []
    for w in sentence:
        out, h = feed_input(model, h, w)
        outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))
    return outs, h

In [30]:
model.eval()
hidden = model.init_hidden(1) 
init_out, init_h = feed_sentence(model, hidden, init_sentence.split(" "))



/tmp/ipykernel_1357831/915584119.py:9: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))


In [31]:
condition_accuracies = defaultdict(int)
condition_counts = defaultdict(int)
correct_pred = 0
sentence_details = []

model.eval()
#Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in test_dataloader:
        out=None
        written = batch['sentence']
        sentence = batch['encoded_sentence']
        correct = batch['encoded_correct']
        wrong = batch['encoded_wrong']
        condition = batch['condition']
        batch_size = sentence.size(0)
        hidden = (init_h[0].expand(-1, batch_size, -1).contiguous(), 
          init_h[1].expand(-1, batch_size, -1).contiguous()) 
        # for sen in sentence : 
        #     print(written[sen])
        #     print(correct[sen])
        #     print(wrong[sen])
        #     hidden = init_h
        for w in range(sentence.shape[1]-1):#update hidden state word by word
            # for w in range(len(sen)):
                
            word = torch.autograd.Variable(sentence[:,w].unsqueeze(0))
            out, hidden = model(word, hidden)
  
        log_probs = torch.nn.functional.log_softmax(out, dim=-1)#s(out.squeeze(0))
        #déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
        correct_log_probs = log_probs[0, torch.arange(batch_size), correct]  # Shape: [512]
        wrong_log_probs = log_probs[0, torch.arange(batch_size), wrong]
        correct_predictions = correct_log_probs >= wrong_log_probs
        for i in range(batch_size):
            cond = condition[i]
            pred = correct_predictions[i].item()  # Convert tensor to Python boolean
            condition_counts[cond] += 1
            condition_accuracies[cond] += pred

            sentence_details.append({
                'sentence': written[i],
                'condition': condition[i],
                'correct_log_prob':correct_log_probs[i],
                'wrong_log_prob':wrong_log_probs[i],
                'model_prefers_correct': pred
            })
final_accuracies = {cond: condition_accuracies[cond] / condition_counts[cond] for cond in condition_accuracies}


for cond, acc in final_accuracies.items():
    print(f"{cond}: Accuracy = {acc * 100:.2f}%")
    
df = pd.DataFrame(sentence_details)


singular singular: Accuracy = 97.20%
singular plural: Accuracy = 72.60%
plural singular: Accuracy = 96.70%
plural plural: Accuracy = 100.00%
